In [1]:
import os
import sys
from pathlib import Path

if 'IS_ROOT_SET' not in globals():
    project_root = Path.cwd().parent.parent.parent.parent
    os.chdir(project_root)
    
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
    
    IS_ROOT_SET = True
    print("Корневая директория инициализирована.")
else:
    print("Инициализация уже была выполнена ранее.")
    

Корневая директория инициализирована.


In [2]:
from mmengine.config import Config
from mmengine.runner import Runner

In [3]:
config = Config.fromfile("mmsegmentation/configs/yandex_config/baseline.py")

config.work_dir = "mmsegmentation/practicum_work/artifacts/results/deeplabv3_experiment1"

config.visualizer.vis_backends = [dict(type='LocalVisBackend')]

config.resume = False
config.load_from = "mmsegmentation/practicum_work/artifacts/deeplabv3_experiment1/epoch_15.pth"

config.test_evaluator = dict(
    type='IoUMetric',
    iou_metrics=['mIoU', 'mDice'],
    output_dir='mmsegmentation/practicum_work/artifacts/results/deeplabv3_experiment1/images' 
)

config.default_hooks.visualization = dict(
    type='SegVisualizationHook', 
    draw=True, 
    interval=1 
)

In [4]:
runner = Runner.from_cfg(config)

06/21 17:10:58 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.12 (main, Aug 15 2025, 14:32:43) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1480012757
    GPU 0: Tesla T4
    CUDA_HOME: /usr
    NVCC: Cuda compilation tools, release 12.2, V12.2.140
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0
    PyTorch: 2.0.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash 6dbeffbae1f23cbbeae17adb7b5b13f1f37c080e)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;arch=compute_37,code=sm_37;-genc

/home/ubuntu/yandex_segmentation_project/mmsegmentation/mmseg/models/backbones/resnet.py:431: UserWarning: DeprecationWarning: pretrained is a deprecated, please use "init_cfg" instead
  warnings.warn('DeprecationWarning: pretrained is a deprecated, '
/home/ubuntu/yandex_segmentation_project/mmsegmentation/mmseg/models/losses/cross_entropy_loss.py:250: UserWarning: Default ``avg_non_ignore`` is False, if you would like to ignore the certain label and average loss over non-ignore labels, which is the same with PyTorch official cross_entropy, set ``avg_non_ignore=True``.
  warnings.warn(


06/21 17:11:00 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
06/21 17:11:00 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                

In [5]:
runner.test()

06/21 17:11:01 - mmengine - WARNING - The prefix is not set in metric class IoUMetric.
Loads checkpoint by local backend from path: mmsegmentation/practicum_work/artifacts/deeplabv3_experiment1/epoch_15.pth
06/21 17:11:02 - mmengine - INFO - Load checkpoint from mmsegmentation/practicum_work/artifacts/deeplabv3_experiment1/epoch_15.pth
06/21 17:11:34 - mmengine - INFO - Iter(test) [20/30]    eta: 0:00:16  time: 0.4854  data_time: 0.0066  memory: 9033  
06/21 17:11:39 - mmengine - INFO - per class results:
06/21 17:11:39 - mmengine - INFO - 
+------------+-------+-------+-------+
|   Class    |  IoU  |  Acc  |  Dice |
+------------+-------+-------+-------+
| background | 95.31 | 96.63 |  97.6 |
|    cat     | 70.57 | 83.35 | 82.74 |
|    dog     | 58.54 | 87.86 | 73.85 |
+------------+-------+-------+-------+
06/21 17:11:39 - mmengine - INFO - Iter(test) [30/30]    aAcc: 95.4700  mIoU: 74.8100  mAcc: 89.2800  mDice: 84.7300  data_time: 0.0153  time: 1.2512


{'aAcc': 95.47, 'mIoU': 74.81, 'mAcc': 89.28, 'mDice': 84.73}

In [6]:
import cv2
import numpy as np

In [7]:
from mmseg.utils.class_names import train_dataset_for_students_classes, train_dataset_for_students_palette

In [8]:
def visualize_dataset_masks(
    image_dir: Path, 
    labels_dir: Path, 
    output_dir: Path,
    img_suffix: str = ".jpg", 
    seg_map_suffix: str = ".png",
    alpha: float = 0.5,
) -> None:
    img_paths = {file.stem: file for file in image_dir.glob(f"*{img_suffix}")}
    mask_paths = {file.stem: file for file in labels_dir.glob(f"*{seg_map_suffix}")}
    
    common_stems = set(img_paths.keys()) & set(mask_paths.keys())
    
    output_dir.mkdir(parents=True, exist_ok=True)
    
    palette = np.array(train_dataset_for_students_palette(), dtype=np.uint8)
    
    for stem in common_stems:
        image_path = img_paths[stem]
        label_path = mask_paths[stem]
        
        image = cv2.imread(str(image_path))
        mask = cv2.imread(str(label_path), cv2.IMREAD_GRAYSCALE)
        
        if image is None or mask is None:
            print(f"Проблема с картинкой или маской: {stem}")
            continue
        
        if image.shape[:2] != mask.shape[:2]:
            print(f"WARNING: Размер картинки и маски не совпадает: {stem}")
            mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)

        color_mask_rgb = palette[mask]
        color_mask_bgr = cv2.cvtColor(color_mask_rgb, cv2.COLOR_RGB2BGR)
        
        overlay = cv2.addWeighted(image, 1.0 - alpha, color_mask_bgr, alpha, 0)
        
        save_path = output_dir / f"{stem}_overlay.jpg"
        cv2.imwrite(filename=str(save_path), img=overlay)            
        
        
        

In [9]:
data_root = Path("mmsegmentation/data/train_dataset_for_students")

In [10]:
visualize_dataset_masks(
    image_dir=data_root / "img" / "test",
    labels_dir=Path("mmsegmentation/practicum_work/artifacts/results/deeplabv3_experiment1/images"),
    output_dir=Path("mmsegmentation/practicum_work/artifacts/results/deeplabv3_experiment1") / "pictures",
)